# 04 — Functional Transformation Pipeline

Demonstrates the functional composition pattern for building PySpark pipelines
applied to **Tasty Bytes Consulting** transaction enrichment:

- `DataFrame.transform(fn)` API for clean left-to-right chaining
- Closures / functions returning functions for parameterised transforms
- Dynamic `when` chains built from configuration lists
- Join-based enrichment encapsulated as transform functions
- Inline validation steps integrated into the pipeline
- ETL timestamp metadata pattern

In [ ]:
import sys; sys.path.insert(0, '.')
from demo_data import create_spark_session, load_all, COMPANY_NAME
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark = create_spark_session('Functional_Pipeline')
spark.sparkContext.setLogLevel('WARN')

data = load_all(spark)
transactions_df = data['transactions']
assets_df       = data['assets']
portfolios_df   = data['portfolios']

print(f'Company : {COMPANY_NAME}')
print(f'Spark   : {spark.version}')
print(f'Rows    : {transactions_df.count()} transactions loaded')

## 1. The `transform` API — simple column derivation

`df.transform(fn)` applies a function `fn(df) -> df` to a DataFrame.
It is syntactic sugar for `fn(df)` but enables clean left-to-right chaining.

Here we add a `notional_usd` column: `quantity * price * fx_rate_to_usd`.

In [ ]:
def with_notional_usd(df: DataFrame) -> DataFrame:
    '''Add notional_usd = quantity * price * fx_rate_to_usd.'''
    return df.withColumn(
        'notional_usd',
        F.round(F.col('quantity') * F.col('price') * F.col('fx_rate_to_usd'), 2),
    )

transactions_df.transform(with_notional_usd).select(
    'txn_id', 'portfolio_id', 'asset_id', 'txn_type',
    'quantity', 'price', 'currency', 'fx_rate_to_usd', 'notional_usd',
).show(10, truncate=False)

## 2. Parameterised transforms — functions returning functions

A closure captures parameters and returns a `DataFrame -> DataFrame` function,
making each transform configurable without losing composability.

In [ ]:
def with_txn_type_label(col: str = 'txn_type', output_col: str = 'txn_type_label'):
    '''Map txn_type codes to descriptive labels.'''
    def _(df: DataFrame) -> DataFrame:
        return df.withColumn(
            output_col,
            F.when(F.col(col) == 'buy',      'Purchase — Long Entry')
             .when(F.col(col) == 'sell',     'Sale — Position Exit')
             .when(F.col(col) == 'dividend', 'Income — Dividend Receipt')
             .when(F.col(col) == 'fee',      'Cost — Management Fee')
             .otherwise('Other'),
        )
    return _


def with_currency_normalized(fx_col: str, price_col: str, output_col: str = 'notional_usd'):
    '''Compute USD notional: quantity * price * fx_rate_to_usd.'''
    def _(df: DataFrame) -> DataFrame:
        return df.withColumn(
            output_col,
            F.round(F.col('quantity') * F.col(price_col) * F.col(fx_col), 2),
        )
    return _


def with_null_filled(column: str, default):
    '''Fill nulls in *column* with *default*.'''
    def _(df: DataFrame) -> DataFrame:
        return df.withColumn(
            column,
            F.when(F.col(column).isNull(), F.lit(default)).otherwise(F.col(column)),
        )
    return _


def with_settlement_lag(
    txn_date_col: str = 'txn_date',
    settlement_date_col: str = 'settlement_date',
    output_col: str = 'settlement_lag_days',
):
    '''Compute calendar days between trade date and settlement date.'''
    def _(df: DataFrame) -> DataFrame:
        return df.withColumn(
            output_col,
            F.datediff(
                F.to_date(F.col(settlement_date_col)),
                F.to_date(F.col(txn_date_col)),
            ),
        )
    return _


def with_etl_timestamp(df: DataFrame) -> DataFrame:
    '''Append pipeline run timestamp as etl_updated_dt.'''
    return df.withColumn('etl_updated_dt', F.current_timestamp())


print('Transform functions defined.')

## 3. Chained pipeline using `.transform()`

Chain transforms left-to-right: fill nulls → notional_usd → txn_type_label → settlement_lag → etl_timestamp.

In [ ]:
enriched_df = (
    transactions_df
    .transform(with_null_filled('fx_rate_to_usd', 1.0))
    .transform(with_currency_normalized('fx_rate_to_usd', 'price', 'notional_usd'))
    .transform(with_txn_type_label())
    .transform(with_settlement_lag())
    .transform(with_etl_timestamp)
)

print('=== Enriched Transactions ===')
enriched_df.select(
    'txn_id', 'portfolio_id', 'asset_id',
    'txn_type', 'txn_type_label',
    'notional_usd', 'settlement_lag_days',
).show(12, truncate=False)

## 4. Dynamic `when` chain from configuration

Build a broker tier classification from a config list at runtime, mirroring
real-world patterns where mappings are driven by config files or lookup tables.

In [ ]:
broker_tier_map = [
    ('Goldman Sachs',       'Tier 1 — Bulge Bracket'),
    ('Morgan Stanley',      'Tier 1 — Bulge Bracket'),
    ('JP Morgan',           'Tier 1 — Bulge Bracket'),
    ('Barclays',            'Tier 2 — Major Bank'),
    ('Deutsche Bank',       'Tier 2 — Major Bank'),
    ('Citigroup',           'Tier 2 — Major Bank'),
    ('UBS',                 'Tier 2 — Major Bank'),
    ('Nordea',              'Tier 3 — Regional'),
    ('Interactive Brokers', 'Tier 3 — Regional'),
    ('Wells Fargo',         'Tier 3 — Regional'),
]


def with_label_from_map(source_col: str, mapping: list, output_col: str = None):
    '''
    Build a when/otherwise chain from a list of (match_value, label) tuples.

    Parameters
    ----------
    source_col : Column to test
    mapping    : [(match_value, label), ...] evaluated in order
    output_col : Output column name (defaults to source_col + "_label")

    Note: broker value "Internal" is always mapped to "Internal".
    '''
    out = output_col or (source_col + '_label')

    def _(df: DataFrame) -> DataFrame:
        if not mapping:
            return df.withColumn(out, F.lit(None).cast(StringType()))
        condition = F.when(F.col(source_col) == mapping[0][0], mapping[0][1])
        for match_val, label in mapping[1:]:
            condition = condition.when(F.col(source_col) == match_val, label)
        condition = condition.when(F.col(source_col) == 'Internal', 'Internal')
        condition = condition.otherwise(F.lit('Unknown'))
        return df.withColumn(out, condition)
    return _


broker_enriched_df = enriched_df.transform(
    with_label_from_map('broker', broker_tier_map, 'broker_tier')
)

print('=== Broker Tier Classification ===')
broker_enriched_df.select('txn_id', 'broker', 'broker_tier').show(15, truncate=False)

## 5. Join-based enrichment as transforms

Encapsulate joins inside transform functions for a readable main pipeline
and independently testable enrichment steps.

In [ ]:
def with_asset_info(assets_df: DataFrame):
    '''Left join on asset_id; bring in asset_class, sector, asset_currency.'''
    asset_lookup = assets_df.select(
        F.col('asset_id'),
        F.col('asset_class'),
        F.col('sector'),
        F.col('currency').alias('asset_currency'),
    )

    def _(df: DataFrame) -> DataFrame:
        return df.join(asset_lookup, on='asset_id', how='left')
    return _


def with_portfolio_info(portfolios_df: DataFrame):
    '''Left join on portfolio_id; bring in strategy, benchmark, base_currency.'''
    portfolio_lookup = portfolios_df.select(
        'portfolio_id', 'strategy', 'benchmark', 'base_currency'
    )

    def _(df: DataFrame) -> DataFrame:
        return df.join(portfolio_lookup, on='portfolio_id', how='left')
    return _


fully_enriched_df = (
    broker_enriched_df
    .transform(with_asset_info(assets_df))
    .transform(with_portfolio_info(portfolios_df))
)

print('=== Fully Enriched Transactions ===')
fully_enriched_df.select(
    'txn_id', 'asset_id', 'asset_class', 'sector', 'asset_currency',
    'portfolio_id', 'strategy', 'benchmark', 'base_currency',
).show(10, truncate=False)

## 6. Validation integrated into the pipeline

Inline validation steps raise an error early rather than propagating
bad data to downstream tables.

In [ ]:
def validate_columns(required: list):
    '''Assert required columns exist; raise ValueError if any are missing.'''
    def _(df: DataFrame) -> DataFrame:
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(f'Pipeline validation failed — missing columns: {missing}')
        print(f'Column validation passed: {required}')
        return df
    return _


def validate_no_nulls_on(columns: list):
    '''Assert none of *columns* contain null values.'''
    def _(df: DataFrame) -> DataFrame:
        for c in columns:
            n = df.filter(F.col(c).isNull()).count()
            if n > 0:
                raise ValueError(f'Validation failed — column {c} has {n} null(s).')
        print(f'No-null validation passed for: {columns}')
        return df
    return _


required_cols = [
    'txn_id', 'portfolio_id', 'asset_id', 'txn_type',
    'notional_usd', 'txn_type_label', 'settlement_lag_days',
    'asset_class', 'sector', 'strategy',
]

validated_df = (
    transactions_df
    .transform(validate_columns(['txn_id', 'portfolio_id', 'asset_id', 'txn_type']))
    .transform(with_null_filled('fx_rate_to_usd', 1.0))
    .transform(with_currency_normalized('fx_rate_to_usd', 'price', 'notional_usd'))
    .transform(with_txn_type_label())
    .transform(with_settlement_lag())
    .transform(with_label_from_map('broker', broker_tier_map, 'broker_tier'))
    .transform(with_asset_info(assets_df))
    .transform(with_portfolio_info(portfolios_df))
    .transform(validate_columns(required_cols))
    .transform(validate_no_nulls_on(['txn_id', 'portfolio_id', 'asset_id']))
    .transform(with_etl_timestamp)
)

print(f'\nFull pipeline completed successfully: {validated_df.count()} rows')
validated_df.select(
    'txn_id', 'portfolio_id', 'asset_id', 'txn_type_label',
    'notional_usd', 'broker_tier', 'asset_class', 'strategy',
).show(10, truncate=False)

In [ ]:
spark.stop()